In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Binary ESI 1 & ESI 5 LightGBM Stage 1 Meta-Classifier (`models/lightbgm_stage1_esi15_meta.ipynb`)

This notebook implements the **Stage 1 Extreme Triage Meta-Classifier** by combining two specialized **Binary LightGBM Models**:
1. **ESI 1 Binary LightGBM Model**: Predicts $P(\text{ESI 1})$ (Immediate Life Threat).
2. **ESI 5 Binary LightGBM Model**: Predicts $P(\text{ESI 5})$ (Non-Urgent Fast-Track).

### System Architecture & Workflow
1. **Full Patient Dataset (ESI 1-5)**: Evaluates $N \approx 560,000$ patient cases mapped into 3 target categories: `"1"` (ESI 1), `"5"` (ESI 5), and `"other"` (ESI 2, 3, 4).
2. **Stratified Data Partitioning First**: Splits dataset into Train (70%), Validation (15%), and Test (15%) splits prior to scaling to prevent data leakage.
3. **Dual-Model Probability Output Stream**: Extracts $P_1 = P(\text{ESI 1})$ and $P_5 = P(\text{ESI 5})$ for every patient, computing $P_{\text{other}} = \max(0, 1 - (P_1 + P_5))$.
4. **Probabilistic Classification Logic**: Classifies patients into `"1"`, `"5"`, or `"other"` based on normalized class probability vectors $[P_1, P_5, P_{\text{other}}]$.
5. **Comprehensive Benchmarking Across Splits**: Evaluates Train, Validation, and Test performance with 3x3 confusion matrices, target class count comparison tables, Accuracy, Per-Class Precision, Per-Class Recall, F1 Score, PR-AUC, and ROC-AUC.
6. **Diagnostic Visualization & CSV Reports**:
   - **Probability Density Plots**: `plots/lightbgm_stage1_esi15_prob_density.png` showing probability distributions across ground truth classes.
   - **Metrics Bar Chart**: `plots/lightbgm_stage1_esi15_metrics_barchart.png`.
   - **CSV Reports**: `reports/lightbgm_stage1_esi15_val_report.csv`, `reports/lightbgm_stage1_esi15_test_report.csv`.
   - **Model Export**: Saved to `deploy/lightbgm_stage1_esi15_meta_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)
has_lgb <- requireNamespace("lightgbm", quietly = TRUE)
if (has_lgb) {
  library(lightgbm)
  cat("LightGBM R package successfully loaded.\n")
} else {
  library(xgboost)
  cat("Note: LightGBM R package not installed. Using XGBoost fallback.\n")
}
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 13 FE Inputs & Map Target ('1', '5', 'other')
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
raw_esi <- as.character(raw_df[[target_col]])
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
# Construct 13 Clinical Feature Engineering flags
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)
# Map target: '1', '5', or 'other' (ESI 2, 3, 4)
df_feng$target_esi15 <- factor(ifelse(raw_esi == "1", "1", ifelse(raw_esi == "5", "5", "other")),
                               levels = c("1", "5", "other"))
initial_rows <- nrow(df_feng)
df_feng <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_feng), nrow(df_feng)))
cat(sprintf("Full ESI 1 / ESI 5 / Other Dataset Ready (Pre-Partitioning): %d total rows x %d cols\n", nrow(df_feng), ncol(df_feng)))
cat("Natural Target Distribution ('1', '5', 'other'):\n")
print(table(df_feng$target_esi15))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning (Train / Validation / Test)
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size
val_size  <- config$training$val_size
# Stratified Test split (15%)
in_train_val <- createDataPartition(df_feng$target_esi15, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]
# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_esi15, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Standardize continuous feature (age)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)
feat_names <- setdiff(names(train_df), "target_esi15")
X_train <- as.matrix(train_df[, feat_names])
X_val   <- as.matrix(val_df[, feat_names])
X_test  <- as.matrix(test_df[, feat_names])
cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))
cat("Train Target Distribution:\n")
print(table(train_df$target_esi15))
cat("\nValidation Target Distribution:\n")
print(table(val_df$target_esi15))
cat("\nTest Target Distribution:\n")
print(table(test_df$target_esi15))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Load / Train Binary ESI 1 & ESI 5 Specialist LightGBM Models
# ---------------------------------------------------------
set.seed(config$training$random_state)
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
model1_path <- file.path(deploy_dir, "lightbgm_feng_esi1_extreme_model.rds")
model5_path <- file.path(deploy_dir, "lightbgm_feng_esi5_extreme_model.rds")
load_or_train_lgb <- function(model_path, target_binary, model_name, sample_ratio_neg = 5.0) {
  if (file.exists(model_path)) {
    cat(sprintf("Loading pre-trained %s model from: %s\n", model_name, model_path))
    obj <- readRDS(model_path)
    return(obj$model)
  } else {
    cat(sprintf("Model artifact not found at %s. Training %s binary model on-the-fly...\n", model_path, model_name))
    y_bin_tr  <- ifelse(train_df$target_esi15 == target_binary, 1, 0)
    y_bin_val <- ifelse(val_df$target_esi15 == target_binary, 1, 0)
    
    pos_idx <- which(y_bin_tr == 1)
    neg_idx <- which(y_bin_tr == 0)
    n_neg_keep <- min(as.integer(length(pos_idx) * sample_ratio_neg), length(neg_idx))
    sub_idx <- sort(c(pos_idx, sample(neg_idx, n_neg_keep)))
    
    if (has_lgb) {
      dtr  <- lgb.Dataset(data = X_train[sub_idx, ], label = y_bin_tr[sub_idx])
      dval <- lgb.Dataset(data = X_val, label = y_bin_val, reference = dtr)
      params <- list(objective = "binary", metric = "binary_logloss", learning_rate = 0.05, num_leaves = 31, max_depth = 6, verbosity = -1)
      m <- lgb.train(params = params, data = dtr, nrounds = 150, valids = list(val = dval), early_stopping_rounds = 20, verbose = -1)
    } else {
      dtr  <- xgb.DMatrix(data = X_train[sub_idx, ], label = y_bin_tr[sub_idx])
      dval <- xgb.DMatrix(data = X_val, label = y_bin_val)
      params <- list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6)
      m <- xgb.train(params = params, data = dtr, nrounds = 150, evals = list(val = dval), early_stopping_rounds = 20, verbose = 0)
    }
    return(m)
  }
}
model_esi1 <- load_or_train_lgb(model1_path, "1", "ESI 1 Specialist LightGBM", sample_ratio_neg = 10.0)
model_esi5 <- load_or_train_lgb(model5_path, "5", "ESI 5 Specialist LightGBM", sample_ratio_neg = 2.0)
cat("Both Stage 1 Binary LightGBM Models (ESI 1 & ESI 5) Ready!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Dual Probability Stream Extraction, Ensemble Classification & CSV Reports
# ---------------------------------------------------------
predict_esi15_meta <- function(model1, model5, X_mat) {
  if (has_lgb) {
    p1 <- predict(model1, newdata = X_mat)
    p5 <- predict(model5, newdata = X_mat)
  } else {
    dmat <- xgb.DMatrix(data = X_mat)
    p1 <- predict(model1, newdata = dmat)
    p5 <- predict(model5, newdata = dmat)
  }
  
  # P(Other) is remaining non-extreme probability
  p_other <- pmax(0, 1 - (p1 + p5))
  
  # Create raw matrix [P1, P5, P_other]
  prob_mat <- cbind(p1 = p1, p5 = p5, p_other = p_other)
  
  # Normalize probabilities across rows so they sum to 1.0
  row_sums <- rowSums(prob_mat)
  prob_norm <- prob_mat / ifelse(row_sums == 0, 1, row_sums)
  colnames(prob_norm) <- c("1", "5", "other")
  
  max_idx <- max.col(prob_norm, ties.method = "first")
  classes <- c("1", "5", "other")
  pred_class <- factor(classes[max_idx], levels = classes)
  
  return(list(prob_norm = prob_norm, pred_class = pred_class, p1 = p1, p5 = p5, p_other = p_other))
}
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_esi15_meta <- function(model1, model5, X_mat, actual_factor, set_name) {
  target_classes <- c("1", "5", "other")
  res <- predict_esi15_meta(model1, model5, X_mat)
  
  prob_norm  <- res$prob_norm
  pred_fac   <- res$pred_class
  act_fac    <- factor(actual_factor, levels = target_classes)
  
  cm  <- confusionMatrix(pred_fac, act_fac)
  acc <- as.numeric(cm$overall["Accuracy"])
  prec_by_class <- cm$byClass[, "Pos Pred Value"]
  rec_by_class  <- cm$byClass[, "Sensitivity"]
  macro_prec    <- mean(prec_by_class, na.rm = TRUE)
  macro_rec     <- mean(rec_by_class,  na.rm = TRUE)
  macro_f1      <- 2 * (macro_prec * macro_rec) / (macro_prec + macro_rec + 1e-15)
  
  pr_auc_by_class <- numeric(3)
  names(pr_auc_by_class) <- target_classes
  for (cls in target_classes) {
    act_bin <- ifelse(act_fac == cls, 1, 0)
    pr_auc_by_class[cls] <- calc_pr_auc(act_bin, prob_norm[, cls])
  }
  macro_pr_auc <- mean(pr_auc_by_class, na.rm = TRUE)
  
  roc_obj <- tryCatch(pROC::multiclass.roc(act_fac, prob_norm), error = function(e) NULL)
  macro_roc_auc <- if (!is.null(roc_obj)) as.numeric(roc_obj$auc) else NA
  
  actual_table <- table(act_fac)
  pred_table   <- table(pred_fac)
  diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = target_classes,
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(prec_by_class, 4),
    Recall       = round(rec_by_class, 4),
    PR_AUC       = round(pr_auc_by_class, 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   STAGE 1 ESI 1 / ESI 5 LIGHTBGM META-CLASSIFIER - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy     : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision      : %.4f (%.2f%%)\n", macro_prec, macro_prec * 100))
  cat(sprintf("  Macro Recall (Sens)  : %.4f (%.2f%%)\n", macro_rec, macro_rec * 100))
  cat(sprintf("  Macro F1 Score       : %.4f\n", macro_f1))
  cat(sprintf("  Macro PR-AUC         : %.4f\n", macro_pr_auc))
  cat(sprintf("  Multi-Class ROC-AUC  : %.4f\n", macro_roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Class Count Comparison & Performance Summary:\n")
  print(report_df)
  cat("\n3x3 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, macro_prec = macro_prec, macro_rec = macro_rec, macro_f1 = macro_f1,
              macro_pr_auc = macro_pr_auc, macro_roc_auc = macro_roc_auc, report_df = report_df, res = res))
}
res_train <- evaluate_esi15_meta(model_esi1, model_esi5, X_train, train_df$target_esi15, "Train")
res_val   <- evaluate_esi15_meta(model_esi1, model_esi5, X_val,   val_df$target_esi15,   "Validation")
res_test  <- evaluate_esi15_meta(model_esi1, model_esi5, X_test,  test_df$target_esi15,  "Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(res_val$report_df,  file = file.path(reports_dir, "lightbgm_stage1_esi15_val_report.csv"),  row.names = FALSE)
write.csv(res_test$report_df, file = file.path(reports_dir, "lightbgm_stage1_esi15_test_report.csv"), row.names = FALSE)
cat("Validation CSV Report written to: reports/lightbgm_stage1_esi15_val_report.csv\n")
cat("Test CSV Report written to:       reports/lightbgm_stage1_esi15_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Probability Distribution Plots & Metrics Bar Chart
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
# 1. PROBABILITY DISTRIBUTION DENSITY PLOTS
test_probs_df <- data.frame(
  Actual_Class = test_df$target_esi15,
  P_ESI1       = res_test$res$prob_norm[, "1"],
  P_ESI5       = res_test$res$prob_norm[, "5"],
  P_Other      = res_test$res$prob_norm[, "other"]
)
test_probs_long <- test_probs_df %>%
  pivot_longer(cols = c("P_ESI1", "P_ESI5", "P_Other"), names_to = "Probability_Type", values_to = "Probability")
p_density <- ggplot(test_probs_long, aes(x = Probability, fill = Actual_Class)) +
  geom_density(alpha = 0.5) +
  facet_wrap(~ Probability_Type, scales = "free_y") +
  theme_minimal() +
  scale_fill_manual(values = c("1" = "#e63946", "5" = "#2a9d8f", "other" = "#457b9d")) +
  labs(title = "Stage 1 LightGBM Model Probability Distributions Across Ground Truth Classes (Test Set)",
       subtitle = "Comparing P(ESI 1), P(ESI 5), and P(Other) for Actual ESI 1, 5, and Other patients",
       x = "Predicted Class Probability", y = "Density", fill = "Actual ESI Class") +
  theme(plot.title = element_text(face = "bold", size = 12), legend.position = "top")
ggsave(file.path(plots_dir, "lightbgm_stage1_esi15_prob_density.png"), plot = p_density, width = 10, height = 4.5, dpi = 300)
cat("Probability Distribution Density Plot saved to: plots/lightbgm_stage1_esi15_prob_density.png\n")
# 2. METRICS COMPARISON BAR CHART
metrics_summary <- data.frame(
  Split     = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy  = c(res_train$acc,          res_val$acc,          res_test$acc),
  Precision = c(res_train$macro_prec,   res_val$macro_prec,   res_test$macro_prec),
  Recall    = c(res_train$macro_rec,    res_val$macro_rec,    res_test$macro_rec),
  F1_Score  = c(res_train$macro_f1,     res_val$macro_f1,     res_test$macro_f1),
  PR_AUC    = c(res_train$macro_pr_auc, res_val$macro_pr_auc, res_test$macro_pr_auc)
)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Precision", "Recall", "F1_Score", "PR_AUC"), names_to = "Metric", values_to = "Score")
p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Train vs. Validation vs. Test Metrics Comparison (Stage 1 ESI 1/5 LightGBM Meta-Classifier)",
       subtitle = "Comparing Accuracy, Precision, Recall, F1 Score, and PR-AUC across splits",
       y = "Metric Value Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")
ggsave(file.path(plots_dir, "lightbgm_stage1_esi15_metrics_barchart.png"), plot = p_bar, width = 9.5, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/lightbgm_stage1_esi15_metrics_barchart.png\n")
print(p_density)
print(p_bar)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Stage 1 ESI 1 / ESI 5 LightGBM Meta Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "lightbgm_stage1_esi15_meta_model.rds")
saveRDS(list(model_esi1 = model_esi1, model_esi5 = model_esi5, preproc = preproc), file = model_path)
cat("Stage 1 ESI 1 / ESI 5 LightGBM Meta Model saved to:", model_path, "\n")